# Algorithm 20 - KLDM PPR q-witness feasibility

This notebook tests the fixed-template q-witness variant of KLDM-PPR step by step:

1. q-only witness fit
2. joint project-only optimization over `(xi_r, xi_v, q)`
3. single PPR kernel against paired renoise-only control
4. full sampler with rollback based on witness feasibility


In [32]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print(f'ROOT={ROOT}')


ROOT=/workspace


In [33]:
import os
import time
import math
import random
import traceback
import importlib
from collections import Counter
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml

import kldmPlus.algorithm19_kldm_ppr_diffcsppp as algo19_backend
import kldmPlus.algorithm20_kldm_ppr_q_witness as algo20_backend
algo19_backend = importlib.reload(algo19_backend)
algo20_backend = importlib.reload(algo20_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import (
    build_diffcsppp_payload_from_syminfo,
    build_symmetry_frame_bridge,
    estimate_semantic_transport_for_reference_order,
    oracle_spacegroup_from_case,
)
from kldmPlus.utils.time import iter_sampling_times

Algorithm19State = algo19_backend.Algorithm19State
Algorithm20Config = algo20_backend.Algorithm20Config
q_only_witness_fit = algo20_backend.q_only_witness_fit
ppr_project_step_q_witness = algo20_backend.ppr_project_step_q_witness
ppr_kernel_q_witness = algo20_backend.ppr_kernel_q_witness
witness_torus_sin_loss = algo20_backend.witness_torus_sin_loss
build_oracle_diffcsppp_payload_from_structure = algo19_backend.build_oracle_diffcsppp_payload_from_structure
c_w_ops = algo19_backend.c_w_ops
center_velocity = algo19_backend.center_velocity
graph_mean_norm = algo19_backend.graph_mean_norm
kldm_clean_fractional_denoiser_Df = algo19_backend.kldm_clean_fractional_denoiser_Df
kldm_ppr_noise_chart = algo19_backend.kldm_ppr_noise_chart
map_model_to_payload_reference_chart = algo19_backend.map_model_to_payload_reference_chart
map_payload_reference_chart_to_model_frame = algo19_backend.map_payload_reference_chart_to_model_frame
torus_rmse = algo19_backend.torus_rmse
wrap01 = algo19_backend.wrap01
wrapdiff = algo19_backend.wrapdiff

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO20_PROFILE', 'laptop').strip().lower()

def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO20_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO20_GRAPH_IDS', '2', '2,3,5'))
ALGO20_T_VALUES = tuple(float(v.strip()) for v in str(profile_default('KLDM_ALGO20_T_VALUES', '0.8,0.5,0.1', '0.8,0.5,0.3,0.1')).split(',') if v.strip())
ALGO20_LAMBDA0_VALUES = tuple(float(v.strip()) for v in str(profile_default('KLDM_ALGO20_LAMBDA0_VALUES', '0.1,1.0', '0.1,1.0,10.0,100.0')).split(',') if v.strip())
ALGO20_M_VALUES = tuple(int(v.strip()) for v in str(profile_default('KLDM_ALGO20_M_VALUES', '1,2', '1,2,4')).split(',') if v.strip())
ALGO20_Q_INIT_MODES = tuple(v.strip() for v in str(profile_default('KLDM_ALGO20_Q_INIT_MODES', 'random,oracle_structure', 'random,oracle_structure')).split(',') if v.strip())
ALGO20_Q_INIT_MODE = str(profile_default('KLDM_ALGO20_Q_INIT_MODE', 'random', 'random'))
ALGO20_Q_ONLY_STEPS = int(profile_default('KLDM_ALGO20_Q_ONLY_STEPS', '100', '150'))
ALGO20_PROJ_STEPS = int(profile_default('KLDM_ALGO20_PROJ_STEPS', '100', '150'))
ALGO20_LR = float(profile_default('KLDM_ALGO20_LR', '1e-2', '1e-2'))
ALGO20_LAMBDA_FLOOR = float(profile_default('KLDM_ALGO20_LAMBDA_FLOOR', '1e-6', '1e-6'))
ALGO20_GRAD_CLIP = float(profile_default('KLDM_ALGO20_GRAD_CLIP', '10.0', '10.0'))
ALGO20_SOFT_ANCHOR_TOL = float(profile_default('KLDM_ALGO20_SOFT_ANCHOR_TOL', '1e-5', '1e-5'))
ALGO20_DENOISER_VARIANT = str(profile_default('KLDM_ALGO20_DENOISER_VARIANT', 'minus', 'minus'))
ALGO20_COORDINATE_SCORE_MODE = str(profile_default('KLDM_ALGO20_COORDINATE_SCORE_MODE', 'direct', 'direct'))
ALGO20_FULL_STEPS = int(profile_default('KLDM_ALGO20_FULL_STEPS', '600', '1000'))
ALGO20_PROJECTION_INTERVAL = int(profile_default('KLDM_ALGO20_PROJECTION_INTERVAL', '50', '50'))
ALGO20_DEFAULT_LAMBDA0 = float(profile_default('KLDM_ALGO20_DEFAULT_LAMBDA0', '0.5', '1.0'))
ALGO20_DEFAULT_M = int(profile_default('KLDM_ALGO20_DEFAULT_M', '1', '2'))
ALGO20_DEFAULT_ANCHOR_MODE = str(profile_default('KLDM_ALGO20_DEFAULT_ANCHOR_MODE', 'soft', 'soft'))
ALGO20_SAMPLER_TAU = float(profile_default('KLDM_ALGO20_SAMPLER_TAU', '0.25', '0.25'))
ALGO20_ROLLBACK = str(profile_default('KLDM_ALGO20_ROLLBACK', 'true', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}

print(f'profile={PROFILE} graphs={GRAPH_IDS} t_values={ALGO20_T_VALUES}')
print(f'lambda0_values={ALGO20_LAMBDA0_VALUES} M_values={ALGO20_M_VALUES} q_init_modes={ALGO20_Q_INIT_MODES}')
print(f'proj_steps={ALGO20_PROJ_STEPS} q_only_steps={ALGO20_Q_ONLY_STEPS} full_steps={ALGO20_FULL_STEPS} projection_interval={ALGO20_PROJECTION_INTERVAL}')


profile=laptop graphs=[2] t_values=(0.8, 0.5, 0.1)
lambda0_values=(0.1, 1.0) M_values=(1, 2) q_init_modes=('random', 'oracle_structure')
proj_steps=100 q_only_steps=100 full_steps=600 projection_interval=50


In [34]:
set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
if not available_graph_ids:
    raise RuntimeError(f'No requested graph ids are present. requested={GRAPH_IDS} available=1..{full_num_graphs}')
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.tolist()
print(f'loaded_graph_ids={available_graph_ids} ptr={ptr}')


mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
loaded_graph_ids=[2] ptr=[0, 16]


In [35]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int


result_tables: dict[str, pd.DataFrame] = {}
payload_cache: dict[int, Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = (
        (edge_index[0] >= start)
        & (edge_index[0] < end)
        & (edge_index[1] >= start)
        & (edge_index[1] < end)
    )
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out


GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(pos=pos, batch=batch_index, num_atoms=num_atoms, edge_node_index=edge_node_index)


def single_graph_batch(case: GraphCase):
    selected = batch.index_select([int(case.graph_idx0)]) if hasattr(batch, 'index_select') else batch[int(case.graph_idx0)]
    if isinstance(selected, list):
        selected = batch.__class__.from_data_list(selected)
    return selected.to(runner.device)


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(
        case.gt_frac,
        case.gt_l_feature,
        case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )


def build_oracle_payload(case: GraphCase):
    if case.graph_id in payload_cache:
        return payload_cache[case.graph_id]
    structure = build_case_structure(case)
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=oracle_spacegroup_from_case(case),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(
        vanilla_structure=structure,
        standardization='refined',
        symprec=1e-2,
        angle_tolerance=5.0,
    )
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    model_to_payload_order = np.asarray(assignment_ref, dtype=int)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'payload_frame': 'pyxtal_refined_standardized',
            'transport_method': bridge.standardized_to_vanilla_method,
            'transport_rmse': float(rmse_ref),
            'model_reference_frac_coords': np.asarray(structure.frac_coords, dtype=float).tolist(),
            'model_to_payload_linear': linear_model_to_std.tolist(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'model_to_payload_order': model_to_payload_order.tolist(),
            'payload_to_model_linear': linear_std_to_model.tolist(),
            'payload_to_model_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'payload_to_model_order': np.asarray(assignment_ref, dtype=int).tolist(),
        },
    )
    payload_cache[case.graph_id] = payload
    return payload


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'min_pair_distance': result.min_pair_distance,
    }


def algo_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def make_noisy_state(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, eps_v, eps_r, r_t = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    state = Algorithm19State(
        f=f_t.detach().clone(),
        v=v_t.detach().clone(),
        l=case.gt_l_feature.detach().clone(),
        atom_types=case.atomic_numbers.detach().clone(),
        node_index=batch_view.batch.detach().clone(),
        edge_node_index=batch_view.edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )
    return state, {'eps_v': eps_v.detach().clone(), 'eps_r': eps_r.detach().clone(), 'r_t': r_t.detach().clone()}


def make_algo_state_from_raw(
    *,
    f: torch.Tensor,
    v: torch.Tensor,
    l: torch.Tensor,
    atom_types: torch.Tensor,
    node_index: torch.Tensor,
    edge_node_index: torch.Tensor,
    t_graph: torch.Tensor,
    t_nodes: torch.Tensor,
) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def paired_renoise_from_f0(
    *,
    model,
    f0: torch.Tensor,
    t_nodes: torch.Tensor,
    node_index: torch.Tensor,
    epsilon_v: torch.Tensor,
    epsilon_r: torch.Tensor,
):
    return model.tdm.sample_noisy_state(
        t=t_nodes,
        f0=f0,
        index=node_index,
        epsilon_v=epsilon_v,
        epsilon_r=epsilon_r,
    )


def sample_paired_eps_like(case: GraphCase, *, seed: int, ref: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    set_seed(int(seed) + 20000 * int(case.graph_id))
    eps_v = torch.randn_like(ref)
    eps_r = torch.randn_like(ref)
    batch_index = torch.zeros(ref.shape[0], device=ref.device, dtype=torch.long)
    eps_v = runner.model.tdm.scatter_center(eps_v, index=batch_index) if hasattr(runner.model.tdm, 'scatter_center') else center_velocity(eps_v, batch_index)
    eps_r = runner.model.tdm.scatter_center(eps_r, index=batch_index) if hasattr(runner.model.tdm, 'scatter_center') else center_velocity(eps_r, batch_index)
    return eps_v, eps_r


def clean_prediction(state: Algorithm19State) -> torch.Tensor:
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=ALGO20_DENOISER_VARIANT,
        coordinate_score_mode=ALGO20_COORDINATE_SCORE_MODE,
    )


def payload_clean_prediction(state: Algorithm19State, payload) -> torch.Tensor:
    return map_model_to_payload_reference_chart(clean_prediction(state), payload)


def make_q_init(case: GraphCase, payload, *, mode: str, seed: int, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    total_dof = int(chart.total_dof)
    if total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'oracle_structure', 'chart_q_ref', 'bootstrap'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'zero', 'zeros'}:
        return torch.zeros((total_dof,), device=device, dtype=dtype)
    if mode_norm in {'random', 'rand'}:
        set_seed(int(seed) + 30000 * int(case.graph_id))
        return torch.rand((total_dof,), device=device, dtype=dtype)
    raise ValueError(f'Unsupported q init mode: {mode!r}')


def witness_stats_from_q(z_payload: torch.Tensor, payload, q: torch.Tensor) -> dict[str, float]:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q.reshape(-1), 1.0) if q.numel() else q.reshape(-1)
    z_sym = chart.expand_q(q_eval, device=z_payload.device, dtype=z_payload.dtype)
    return {
        'witness_sin': float(witness_torus_sin_loss(z_payload, z_sym).detach().item()),
        'witness_rmse_payload': float(torus_rmse(z_payload, z_sym).detach().item()),
    }


def q_only_fit_from_clean(clean_f: torch.Tensor, payload, *, case: GraphCase, q_init_mode: str, seed: int, q_init: torch.Tensor | None = None):
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    q_seed = q_init if q_init is not None else make_q_init(
        case,
        payload,
        mode=q_init_mode,
        seed=seed,
        dtype=clean_f.dtype,
        device=clean_f.device,
    )
    before = witness_stats_from_q(z_payload, payload, q_seed)
    fit = q_only_witness_fit(
        z_payload=z_payload,
        payload=payload,
        q_init=q_seed,
        q_init_mode=q_init_mode,
        steps=int(ALGO20_Q_ONLY_STEPS),
        lr=float(ALGO20_LR),
        grad_clip=float(ALGO20_GRAD_CLIP),
    )
    return {
        'z_payload': z_payload,
        'q_seed': q_seed,
        'before': before,
        'fit': fit,
    }


def config20(*, M: int, lambda0: float, q_init_mode: str = ALGO20_Q_INIT_MODE) -> Algorithm20Config:
    return Algorithm20Config(
        M=int(M),
        proj_steps=int(ALGO20_PROJ_STEPS),
        lr=float(ALGO20_LR),
        lambda_noise=float(lambda0),
        lambda_floor=float(ALGO20_LAMBDA_FLOOR),
        grad_clip=float(ALGO20_GRAD_CLIP),
        anchor_mode=str(ALGO20_DEFAULT_ANCHOR_MODE),
        denoiser_variant=str(ALGO20_DENOISER_VARIANT),
        coordinate_score_mode=str(ALGO20_COORDINATE_SCORE_MODE),
        soft_anchor_tol=float(ALGO20_SOFT_ANCHOR_TOL),
        q_init_mode=str(q_init_mode),
        q_only_steps=int(ALGO20_Q_ONLY_STEPS),
    )


loaded_graphs= [2] sg= [4]


## 1. q-only witness fit

In [36]:
q_only_rows = []
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in ALGO20_T_VALUES:
        for q_init_mode in ALGO20_Q_INIT_MODES:
            try:
                state, _ = make_noisy_state(case, t_value=float(t_value), seed=int(SAMPLE_SEED))
                clean_f = clean_prediction(state)
                q_debug = q_only_fit_from_clean(clean_f, payload, case=case, q_init_mode=q_init_mode, seed=SAMPLE_SEED)
                q_only_rows.append({
                    'test': 'algo20_q_only_witness_fit',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't_value': float(t_value),
                    'q_init_mode': q_init_mode,
                    'c_model_clean': float(c_w_ops(clean_f, payload).detach().item()),
                    'q_only_witness_sin_before': float(q_debug['before']['witness_sin']),
                    'q_only_witness_rmse_before': float(q_debug['before']['witness_rmse_payload']),
                    'q_only_witness_sin_after': float(q_debug['fit']['witness_sin']),
                    'q_only_witness_rmse_after': float(q_debug['fit']['witness_rmse_payload']),
                    'q_norm': float(torch.linalg.norm(q_debug['fit']['q_star']).detach().item()) if q_debug['fit']['q_star'].numel() else 0.0,
                    'PASS': bool(q_debug['fit']['witness_sin'] <= q_debug['before']['witness_sin'] + 1e-12),
                    'status': 'PASS' if bool(q_debug['fit']['witness_sin'] <= q_debug['before']['witness_sin'] + 1e-12) else 'WARN',
                })
            except Exception as exc:
                q_only_rows.append(error_row(
                    'algo20_q_only_witness_fit',
                    case.graph_id,
                    exc,
                    'q_only_witness_fit',
                    space_group=case.requested_sg,
                    t_value=float(t_value),
                    q_init_mode=q_init_mode,
                ))

q_only_df = dataframe_result('algo20_q_only_witness_fit', q_only_rows)
q_only_df = q_only_df.sort_values(['graph', 't_value', 'q_init_mode']).reset_index(drop=True)
display(q_only_df)


,test,graph,space_group,t_value,q_init_mode,c_model_clean,q_only_witness_sin_before,q_only_witness_rmse_before,q_only_witness_sin_after,q_only_witness_rmse_after,q_norm,PASS,status
0,algo20_q_only_witness_fit,2,4,0.1,oracle_structure,6.588282e-07,0.000082,0.002882,0.000007,0.000812,2.798700,True,PASS
1,algo20_q_only_witness_fit,2,4,0.1,random,6.588282e-07,0.626533,0.328098,0.000042,0.002074,2.794641,True,PASS
2,algo20_q_only_witness_fit,2,4,0.5,oracle_structure,1.990553e-05,0.000581,0.007674,0.000196,0.004462,2.809559,True,PASS
3,algo20_q_only_witness_fit,2,4,0.5,random,1.990553e-05,0.625974,0.327701,0.000232,0.004847,2.805264,True,PASS
4,algo20_q_only_witness_fit,2,4,0.8,oracle_structure,2.443296e-03,0.059098,0.083387,0.022995,0.049431,2.786128,True,PASS
5,algo20_q_only_witness_fit,2,4,0.8,random,2.443296e-03,0.605492,0.317700,0.023022,0.049460,2.782152,True,PASS


## 2. Joint project-only over `(xi_r, xi_v, q)`

In [37]:
project_rows = []
project_trace_rows = []
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    for t_value in ALGO20_T_VALUES:
        for lambda0 in ALGO20_LAMBDA0_VALUES:
            try:
                state, _ = make_noisy_state(case, t_value=float(t_value), seed=int(SAMPLE_SEED))
                clean_f = clean_prediction(state)
                q_seed = make_q_init(case, payload, mode=ALGO20_Q_INIT_MODE, seed=SAMPLE_SEED, dtype=state.f.dtype, device=state.f.device)
                witness_before = witness_stats_from_q(map_model_to_payload_reference_chart(clean_f, payload), payload, q_seed)
                project = ppr_project_step_q_witness(
                    state=state,
                    payload=payload,
                    model=runner.model,
                    config=config20(M=1, lambda0=float(lambda0), q_init_mode=ALGO20_Q_INIT_MODE),
                    q_init=q_seed,
                )
                tail = project.logs[-1] if project.logs else {}
                project_rows.append({
                    'test': 'algo20_project_only_feasibility',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    't_value': float(t_value),
                    'lambda0': float(lambda0),
                    'q_init_mode': ALGO20_Q_INIT_MODE,
                    'c_before_witness_sin': float(witness_before['witness_sin']),
                    'c_after_witness_sin': float(project.witness_sin),
                    'witness_rmse_payload': float(project.witness_rmse_payload),
                    'xi_r_norm': float(tail.get('xi_r_norm', float('nan'))),
                    'xi_v_norm': float(tail.get('xi_v_norm', float('nan'))),
                    'q_norm': float(tail.get('q_norm', float('nan'))),
                    'q_step_norm': float(tail.get('q_step_norm', float('nan'))),
                    'q_grad_norm': float(tail.get('q_grad_norm', float('nan'))),
                    'lambda_eff': float(project.lambda_eff),
                    'PASS': bool(project.witness_sin <= witness_before['witness_sin'] + 1e-12),
                    'status': 'PASS' if bool(project.witness_sin <= witness_before['witness_sin'] + 1e-12) else 'WARN',
                })
                for log in project.logs:
                    project_trace_rows.append({
                        'test': 'algo20_project_trace',
                        'graph': case.graph_id,
                        'space_group': case.requested_sg,
                        't_value': float(t_value),
                        'lambda0': float(lambda0),
                        **log,
                        'PASS': True,
                        'status': 'INFO',
                    })
            except Exception as exc:
                project_rows.append(error_row(
                    'algo20_project_only_feasibility',
                    case.graph_id,
                    exc,
                    'project_only_feasibility',
                    space_group=case.requested_sg,
                    t_value=float(t_value),
                    lambda0=float(lambda0),
                ))

project_df = dataframe_result('algo20_project_only_feasibility', project_rows)
project_df = project_df.sort_values(['graph', 't_value', 'lambda0']).reset_index(drop=True)
display(project_df)

project_trace_df = dataframe_result('algo20_project_trace', project_trace_rows)
project_trace_df = project_trace_df.sort_values(['graph', 't_value', 'lambda0', 'step']).reset_index(drop=True) if not project_trace_df.empty else project_trace_df
display(project_trace_df)


KeyboardInterrupt: 

## 3. Single PPR kernel against paired renoise-only control

In [ ]:
def fixed_t_kernel_debug_row(
    case: GraphCase,
    *,
    t_value: float,
    lambda0: float,
    M: int,
    seed: int = SAMPLE_SEED,
) -> dict[str, Any]:
    payload = build_oracle_payload(case)
    state, _ = make_noisy_state(case, t_value=float(t_value), seed=int(seed))
    clean_before = clean_prediction(state)
    q_before = q_only_fit_from_clean(clean_before, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed)

    eps_v, eps_r = sample_paired_eps_like(case, seed=int(seed), ref=clean_before)
    f_renoise, v_renoise, *_ = paired_renoise_from_f0(
        model=runner.model,
        f0=clean_before,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        epsilon_v=eps_v,
        epsilon_r=eps_r,
    )
    renoise_state = replace(state, f=f_renoise.detach().clone(), v=v_renoise.detach().clone())
    clean_renoise = clean_prediction(renoise_state)
    renoise_fit = q_only_fit_from_clean(clean_renoise, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed)

    kernel = ppr_kernel_q_witness(
        state=state,
        payload=payload,
        model=runner.model,
        config=config20(M=int(M), lambda0=float(lambda0), q_init_mode=ALGO20_Q_INIT_MODE),
        q_init=None,
        epsilon_sequence=[(eps_v, eps_r)] + [None for _ in range(max(int(M) - 1, 0))],
    )
    kernel_tail = kernel.logs[-1] if kernel.logs else {}
    clean_ppr_redenoise = clean_prediction(kernel.state)
    ppr_fit = q_only_fit_from_clean(
        clean_ppr_redenoise,
        payload,
        case=case,
        q_init_mode=ALGO20_Q_INIT_MODE,
        seed=seed,
        q_init=kernel.q_star,
    )
    accepted = bool(kernel_tail.get('soft_anchor_feasible', False))

    return {
        'test': 'algo20_single_kernel_debug',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        't_value': float(t_value),
        'lambda0': float(lambda0),
        'M': int(M),
        'q_init_mode': ALGO20_Q_INIT_MODE,
        'c_before_witness_sin': float(q_before['fit']['witness_sin']),
        'c_after_witness_sin': float(kernel_tail.get('c_after_witness_sin', float('nan'))),
        'witness_rmse_payload': float(kernel_tail.get('witness_rmse_payload', float('nan'))),
        'xi_r_norm': float(kernel_tail.get('project_logs', [{}])[-1].get('xi_r_norm', float('nan')) if kernel_tail.get('project_logs') else float('nan')),
        'xi_v_norm': float(kernel_tail.get('project_logs', [{}])[-1].get('xi_v_norm', float('nan')) if kernel_tail.get('project_logs') else float('nan')),
        'q_norm': float(kernel_tail.get('q_norm', float('nan'))),
        'q_step_norm': float(kernel_tail.get('q_step_norm', float('nan'))),
        'q_grad_norm': float(kernel_tail.get('project_logs', [{}])[-1].get('q_grad_norm', float('nan')) if kernel_tail.get('project_logs') else float('nan')),
        'c_after_redenoise_witness': float(ppr_fit['fit']['witness_sin']),
        'renoise_only_witness_control': float(renoise_fit['fit']['witness_sin']),
        'ppr_beats_renoise': bool(ppr_fit['fit']['witness_sin'] < renoise_fit['fit']['witness_sin'] + 1e-12),
        'soft_anchor_feasible': bool(kernel_tail.get('soft_anchor_feasible', False)),
        'accepted': accepted,
        'frac_rmse_renoise_only': float(evaluate_arrays(case, clean_renoise, state.l, case.atomic_numbers)['frac_rmse']),
        'frac_rmse_ppr_redenoise': float(evaluate_arrays(case, clean_ppr_redenoise, state.l, case.atomic_numbers)['frac_rmse']),
        'PASS': bool(ppr_fit['fit']['witness_sin'] < renoise_fit['fit']['witness_sin'] + 1e-12),
        'status': 'PASS' if bool(ppr_fit['fit']['witness_sin'] < renoise_fit['fit']['witness_sin'] + 1e-12) else 'WARN',
    }


kernel_rows = []
for case in GRAPH_CASES:
    for t_value in ALGO20_T_VALUES:
        for lambda0 in ALGO20_LAMBDA0_VALUES:
            for M in ALGO20_M_VALUES:
                try:
                    kernel_rows.append(fixed_t_kernel_debug_row(case, t_value=float(t_value), lambda0=float(lambda0), M=int(M)))
                except Exception as exc:
                    kernel_rows.append(error_row(
                        'algo20_single_kernel_debug',
                        case.graph_id,
                        exc,
                        'single_kernel_debug',
                        space_group=case.requested_sg,
                        t_value=float(t_value),
                        lambda0=float(lambda0),
                        M=int(M),
                    ))

kernel_debug_df = dataframe_result('algo20_single_kernel_debug', kernel_rows)
kernel_debug_df = kernel_debug_df.sort_values(['graph', 't_value', 'lambda0', 'M']).reset_index(drop=True)
display(kernel_debug_df)


,test,graph,space_group,t_value,lambda0,M,q_init_mode,c_before_witness_sin,c_after_witness_sin,witness_rmse_payload,...,q_grad_norm,c_after_redenoise_witness,renoise_only_witness_control,ppr_beats_renoise,soft_anchor_feasible,accepted,frac_rmse_renoise_only,frac_rmse_ppr_redenoise,PASS,status
0,algo20_single_kernel_debug,2,4,0.1,0.1,1,random,0.000042,0.000024,0.001575,...,0.006280,0.000005,0.000043,True,False,False,0.004170,0.004119,True,PASS
1,algo20_single_kernel_debug,2,4,0.1,0.1,2,random,0.000042,0.000003,0.000513,...,0.000146,0.000008,0.000043,True,True,True,0.004170,0.004405,True,PASS
2,algo20_single_kernel_debug,2,4,0.1,1.0,1,random,0.000042,0.000026,0.001608,...,0.006136,0.000006,0.000043,True,False,False,0.004170,0.004142,True,PASS
3,algo20_single_kernel_debug,2,4,0.1,1.0,2,random,0.000042,0.000005,0.000713,...,0.000123,0.000009,0.000043,True,True,True,0.004170,0.004369,True,PASS
4,algo20_single_kernel_debug,2,4,0.5,0.1,1,random,0.000232,0.000050,0.002249,...,0.003741,0.000054,0.000115,True,False,False,0.006945,0.006975,True,PASS
5,algo20_single_kernel_debug,2,4,0.5,0.1,2,random,0.000232,0.000002,0.000468,...,0.000098,0.000149,0.000115,False,True,True,0.006945,0.008803,False,WARN
6,algo20_single_kernel_debug,2,4,0.5,1.0,1,random,0.000232,0.000052,0.002306,...,0.003457,0.000053,0.000115,True,False,False,0.006945,0.006331,True,PASS
7,algo20_single_kernel_debug,2,4,0.5,1.0,2,random,0.000232,0.000005,0.000704,...,0.000185,0.000130,0.000115,False,True,True,0.006945,0.008572,False,WARN
8,algo20_single_kernel_debug,2,4,0.8,0.1,1,random,0.023022,0.001695,0.013130,...,0.009018,0.013449,0.021792,True,False,False,0.111497,0.090111,True,PASS
9,algo20_single_kernel_debug,2,4,0.8,0.1,2,random,0.023022,0.001914,0.013940,...,0.004709,0.002720,0.021792,True,False,False,0.111497,0.080301,True,PASS


## 4. Full sampler with rollback

In [ ]:

def run_pc_sampler_case(case: GraphCase, *, n_steps: int = ALGO20_FULL_STEPS, tau: float = ALGO20_SAMPLER_TAU, seed: int = SAMPLE_SEED) -> dict[str, Any]:
    set_seed(int(seed) + 10000 * int(case.graph_id) + 17)
    graph_batch = single_graph_batch(case)
    payload = build_oracle_payload(case)
    started = time.perf_counter()
    with torch.no_grad():
        f_t, v_t, l_t, h_t = runner.model.sample_CSP_algorithm4(
            n_steps=500,#int(n_steps),
            batch=graph_batch,
            tau=float(tau),
        )
    elapsed_s = float(time.perf_counter() - started)
    endpoint = evaluate_arrays(case, f_t, l_t.reshape(-1), h_t)
    q_eval = q_only_fit_from_clean(f_t, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed)
    return {
        'test': 'algo20_sampler_compare',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'method': f'pc_{int(n_steps)}',
        'n_steps': int(n_steps),
        'projection_interval': np.nan,
        'M': 0,
        'lambda0': np.nan,
        'schedule_label': 'none',
        'rollback': False,
        'runtime_s': elapsed_s,
        'final_c_model': float(c_w_ops(f_t, payload).detach().item()),
        'final_witness_sin': float(q_eval['fit']['witness_sin']),
        'frac_rmse': float(endpoint['frac_rmse']),
        'rmse': float(endpoint['rmse']) if endpoint['rmse'] is not None else float('nan'),
        'match': bool(endpoint['match']),
        'valid': bool(endpoint['valid']),
        'PASS': True,
        'status': 'INFO',
    }


PHASE4_LATE_START_HIGH = 0.5
PHASE4_LATE_START_MID = 0.3
PHASE4_LATE_START_LOW = 0.05
PHASE4_LATE_INTERVAL_LOW = 25
PHASE4_MID_IMPROVEMENT_RATIO = 1.50
PHASE4_LOW_IMPROVEMENT_RATIO = 1.05
PHASE4_VERY_LATE_IMPROVEMENT_RATIO = 1.01


def phase4_schedule(*, t_value: float, base_projection_interval: int = ALGO20_PROJECTION_INTERVAL) -> dict[str, Any]:
    t = float(t_value)
    interval = int(base_projection_interval)
    schedule = {
        'use_ppr': False,
        'projection_interval': int(interval),
        'M': 0,
        'lambda0': 0.0,
        'anchor_mode': 'soft',
        'accept_mode': 'skip',
        'min_improvement_ratio': float('inf'),
        'schedule_label': f'late_soft_skip_t={t:.3f}',
    }
    if t > PHASE4_LATE_START_HIGH:
        schedule.update({
            'use_ppr': False,
            'schedule_label': f'late_soft_skip_high_t={t:.3f}',
        })
        return schedule
    if PHASE4_LATE_START_HIGH >= t > PHASE4_LATE_START_MID:
        schedule.update({
            'use_ppr': True,
            'projection_interval': int(interval),
            'M': 1,
            'lambda0': 1.0,
            'anchor_mode': 'soft',
            'accept_mode': 'improve',
            'min_improvement_ratio': float(PHASE4_MID_IMPROVEMENT_RATIO),
            'schedule_label': f'late_soft_mid_t={t:.3f}_M=1_lambda0=1.0_ratio={PHASE4_MID_IMPROVEMENT_RATIO:.2f}',
        })
        return schedule
    if PHASE4_LATE_START_MID >= t > PHASE4_LATE_START_LOW:
        schedule.update({
            'use_ppr': True,
            'projection_interval': int(min(interval, PHASE4_LATE_INTERVAL_LOW)),
            'M': 1,
            'lambda0': 0.3,
            'anchor_mode': 'soft',
            'accept_mode': 'improve',
            'min_improvement_ratio': float(PHASE4_LOW_IMPROVEMENT_RATIO),
            'schedule_label': f'late_soft_low_t={t:.3f}_M=1_lambda0=0.3_ratio={PHASE4_LOW_IMPROVEMENT_RATIO:.2f}',
        })
        return schedule
    schedule.update({
        'use_ppr': True,
        'projection_interval': int(min(interval, PHASE4_LATE_INTERVAL_LOW)),
        'M': 1,
        'lambda0': 0.1,
        'anchor_mode': 'soft',
        'accept_mode': 'improve_or_feasible',
        'min_improvement_ratio': float(PHASE4_VERY_LATE_IMPROVEMENT_RATIO),
        'schedule_label': f'late_soft_final_t={t:.3f}_M=1_lambda0=0.1_ratio={PHASE4_VERY_LATE_IMPROVEMENT_RATIO:.2f}',
    })
    return schedule


def _accept_phase4_projection(*, before_witness_sin: float, after_witness_sin: float, soft_anchor_feasible: bool, rollback: bool, accept_mode: str, min_improvement_ratio: float) -> bool:
    if not rollback:
        return True
    denom = max(float(after_witness_sin), 1e-12)
    improvement_ratio = float(before_witness_sin) / denom
    if accept_mode == 'skip':
        return False
    if accept_mode == 'improve':
        return bool(improvement_ratio >= float(min_improvement_ratio))
    if accept_mode == 'improve_or_feasible':
        return bool(soft_anchor_feasible or improvement_ratio >= float(min_improvement_ratio))
    if accept_mode == 'feasible':
        return bool(soft_anchor_feasible)
    raise ValueError(f'Unsupported accept_mode={accept_mode!r}')


def run_ppr_sampler_case(
    case: GraphCase,
    *,
    n_steps: int = ALGO20_FULL_STEPS,
    projection_interval: int = ALGO20_PROJECTION_INTERVAL,
    M: int = ALGO20_DEFAULT_M,
    lambda0: float = ALGO20_DEFAULT_LAMBDA0,
    seed: int = SAMPLE_SEED,
    rollback: bool = ALGO20_ROLLBACK,
    use_schedule: bool = True,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    set_seed(int(seed) + 10000 * int(case.graph_id) + 17)
    graph_batch = single_graph_batch(case)
    payload = build_oracle_payload(case)
    state = runner.model._prepare_csp_sampling(
        batch=graph_batch,
        n_steps=int(n_steps),
        t_start=1.0,
        t_final=1e-6,
    )

    projection_trace = []
    q_live = None
    started = time.perf_counter()
    scheduled_M = int(M)
    scheduled_lambda0 = float(lambda0)
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        with torch.no_grad():
            preds_curr = runner.model.score_network(
                t=times.now.graph,
                pos=state['f_t'],
                v=state['v_t'],
                h=state['a_t'],
                l=state['l_t'],
                node_index=state['node_index'],
                edge_node_index=state['edge_node_index'],
            )
            state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_predictor(
                t=times.now.nodes,
                f_t=state['f_t'],
                v_t=state['v_t'],
                pred_v=preds_curr['v'],
                index=state['node_index'],
                dt=times.dt,
            )
            if times.t_next_float >= 1e-3:
                preds_next = runner.model.score_network(
                    t=times.next.graph,
                    pos=state['f_t'],
                    v=state['v_t'],
                    h=state['a_t'],
                    l=state['l_t'],
                    node_index=state['node_index'],
                    edge_node_index=state['edge_node_index'],
                )
                state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_corrector(
                    t=times.next.nodes,
                    f_t=state['f_t'],
                    v_t=state['v_t'],
                    pred_v=preds_next['v'],
                    dt=times.dt,
                    index=state['node_index'],
                    tau=float(ALGO20_SAMPLER_TAU),
                )
                state['l_t'] = runner.model._reverse_lattice_sampling_step(
                    t=times.next.lattice,
                    x_t=state['l_t'],
                    pred=preds_next['l'],
                    dt=times.dt,
                    num_atoms=state['batch'].num_atoms,
                )

        scheduled = phase4_schedule(t_value=float(times.t_next_float), base_projection_interval=int(projection_interval)) if use_schedule else {
            'use_ppr': True,
            'projection_interval': int(projection_interval),
            'M': int(M),
            'lambda0': float(lambda0),
            'anchor_mode': 'soft',
            'accept_mode': 'feasible',
            'min_improvement_ratio': 1.0,
            'schedule_label': 'constant',
        }
        scheduled_M = int(scheduled['M'])
        scheduled_lambda0 = float(scheduled['lambda0'])
        schedule_label = str(scheduled['schedule_label'])
        if (not bool(scheduled['use_ppr'])) or (step_idx % int(scheduled['projection_interval']) != 0):
            continue

        algo_state = make_algo_state_from_raw(
            f=state['f_t'],
            v=state['v_t'],
            l=state['l_t'],
            atom_types=state['a_t'],
            node_index=state['node_index'],
            edge_node_index=state['edge_node_index'],
            t_graph=times.next.graph,
            t_nodes=times.next.nodes,
        )
        clean_before = clean_prediction(algo_state)
        before_fit = q_only_fit_from_clean(clean_before, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed, q_init=q_live)
        kernel = ppr_kernel_q_witness(
            state=algo_state,
            payload=payload,
            model=runner.model,
            config=replace(config20(M=scheduled_M, lambda0=scheduled_lambda0, q_init_mode=ALGO20_Q_INIT_MODE), anchor_mode=str(scheduled['anchor_mode'])),
            q_init=q_live,
        )
        kernel_tail = kernel.logs[-1] if kernel.logs else {}
        soft_anchor_feasible = bool(kernel_tail.get('soft_anchor_feasible', False))
        accepted = _accept_phase4_projection(
            before_witness_sin=float(before_fit['fit']['witness_sin']),
            after_witness_sin=float(kernel_tail.get('c_after_witness_sin', float('inf'))),
            soft_anchor_feasible=soft_anchor_feasible,
            rollback=bool(rollback),
            accept_mode=str(scheduled['accept_mode']),
            min_improvement_ratio=float(scheduled['min_improvement_ratio']),
        )
        if accepted:
            state['f_t'] = kernel.state.f.detach().clone()
            state['v_t'] = kernel.state.v.detach().clone()
            q_live = None if kernel.q_star is None else kernel.q_star.detach().clone()

        clean_after = clean_prediction(make_algo_state_from_raw(
            f=state['f_t'],
            v=state['v_t'],
            l=state['l_t'],
            atom_types=state['a_t'],
            node_index=state['node_index'],
            edge_node_index=state['edge_node_index'],
            t_graph=times.next.graph,
            t_nodes=times.next.nodes,
        ))
        after_fit = q_only_fit_from_clean(clean_after, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed, q_init=q_live)
        improvement_ratio = float(before_fit['fit']['witness_sin']) / max(float(kernel_tail.get('c_after_witness_sin', float('inf'))), 1e-12)
        projection_trace.append({
            'test': 'algo20_sampler_projection_trace',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'method': f'ppr_kldm_{int(n_steps)}_late_soft_sched',
            'step': int(step_idx),
            't_next': float(times.t_next_float),
            'M': int(scheduled_M),
            'lambda0': float(scheduled_lambda0),
            'schedule_label': schedule_label,
            'rollback': bool(rollback),
            'accept_mode': str(scheduled['accept_mode']),
            'min_improvement_ratio': float(scheduled['min_improvement_ratio']),
            'c_before_witness': float(before_fit['fit']['witness_sin']),
            'c_after_project_anchor': float(kernel_tail.get('c_after_witness_sin', float('nan'))),
            'c_after_redenoise': float(after_fit['fit']['witness_sin']),
            'ppr_beats_renoise_proxy': bool(float(after_fit['fit']['witness_sin']) < float(before_fit['fit']['witness_sin'])),
            'witness_rmse_payload': float(kernel_tail.get('witness_rmse_payload', float('nan'))),
            'soft_anchor_feasible': bool(soft_anchor_feasible),
            'accepted': bool(accepted),
            'improvement_ratio': float(improvement_ratio),
            'q_norm': float(kernel_tail.get('q_norm', float('nan'))),
            'status': 'INFO',
            'PASS': True,
        })

    elapsed_s = float(time.perf_counter() - started)
    if state['restore_training']:
        state['score_network'].train()

    endpoint = evaluate_arrays(case, state['f_t'], state['l_t'].reshape(-1), state['a_t'])
    final_fit = q_only_fit_from_clean(state['f_t'], payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed, q_init=q_live)
    accepted_count = sum(bool(row['accepted']) for row in projection_trace)
    feasible_count = sum(bool(row['soft_anchor_feasible']) for row in projection_trace)
    schedule_labels = sorted(set(str(row.get('schedule_label', '')) for row in projection_trace))
    compare_row = {
        'test': 'algo20_sampler_compare',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'method': f'ppr_kldm_{int(n_steps)}_late_soft_sched',
        'n_steps': int(n_steps),
        'projection_interval': int(projection_interval),
        'M': int(max((row['M'] for row in projection_trace), default=scheduled_M)),
        'lambda0': float(max((row['lambda0'] for row in projection_trace), default=scheduled_lambda0)),
        'schedule_label': ' | '.join(schedule_labels) if schedule_labels else 'no_ppr_triggered',
        'rollback': bool(rollback),
        'runtime_s': elapsed_s,
        'projection_count': len(projection_trace),
        'accepted_fraction': float(accepted_count / max(len(projection_trace), 1)),
        'soft_anchor_feasible_fraction': float(feasible_count / max(len(projection_trace), 1)),
        'final_c_model': float(c_w_ops(state['f_t'], payload).detach().item()),
        'final_witness_sin': float(final_fit['fit']['witness_sin']),
        'frac_rmse': float(endpoint['frac_rmse']),
        'rmse': float(endpoint['rmse']) if endpoint['rmse'] is not None else float('nan'),
        'match': bool(endpoint['match']),
        'valid': bool(endpoint['valid']),
        'PASS': True,
        'status': 'INFO',
    }
    return compare_row, projection_trace

sampler_rows = []
projection_trace_rows = []
for case in GRAPH_CASES:
    try:
        sampler_rows.append(run_pc_sampler_case(case))
    except Exception as exc:
        sampler_rows.append(error_row('algo20_sampler_compare', case.graph_id, exc, 'pc_sampler', space_group=case.requested_sg, method=f'pc_{int(ALGO20_FULL_STEPS)}'))
    try:
        compare_row, trace_rows = run_ppr_sampler_case(
            case,
            n_steps=int(ALGO20_FULL_STEPS),
            projection_interval=int(ALGO20_PROJECTION_INTERVAL),
            M=int(ALGO20_DEFAULT_M),
            lambda0=float(ALGO20_DEFAULT_LAMBDA0),
            rollback=bool(ALGO20_ROLLBACK),
            use_schedule=True,
        )
        sampler_rows.append(compare_row)
        projection_trace_rows.extend(trace_rows)
    except Exception as exc:
        sampler_rows.append(error_row(
            'algo20_sampler_compare',
            case.graph_id,
            exc,
            'ppr_sampler',
            space_group=case.requested_sg,
            method=f'ppr_kldm_{int(ALGO20_FULL_STEPS)}_late_soft_sched',
            M=int(ALGO20_DEFAULT_M),
            lambda0=float(ALGO20_DEFAULT_LAMBDA0),
        ))

sampler_compare_df = dataframe_result('algo20_sampler_compare', sampler_rows)
sampler_compare_df = sampler_compare_df.sort_values(['graph', 'method']).reset_index(drop=True)
display(sampler_compare_df)

sampler_projection_trace_df = dataframe_result('algo20_sampler_projection_trace', projection_trace_rows)
sampler_projection_trace_df = sampler_projection_trace_df.sort_values(['graph', 'step']).reset_index(drop=True) if not sampler_projection_trace_df.empty else sampler_projection_trace_df
display(sampler_projection_trace_df)


KeyboardInterrupt: 

## 5. Oracle-template constrained initialization

This section tests the constrained-basin claim directly.

For this notebook we treat the **CrystalFormer-style prior as oracle**:

1. take the oracle template / oracle occupied Wyckoff structure as if CrystalFormer had proposed it,
2. sample `q0` on that template,
3. build `F0 = Phi_T(q0)` in payload frame and map it to model frame,
4. forward-noise with the native KLDM/TDM kernel,
5. run reverse dynamics with and without Algorithm 20 PPR.

This is the cleanest proof target for the oracle-template PPR mechanism, because the chain starts inside the right template basin before noising.

So conceptually this section is testing the pipeline:

- `A -> G` from CSPML (oracle here)
- CrystalFormer proposes a template-compatible seed `(T, q, F_seed, L_seed)`
- Algorithm 20 does local KLDM-PPR refinement from that prior

For now we assume CrystalFormer's proposed template-compatible prior is oracle, so we can test whether that kind of prior is good enough for the PPR mechanism to work.


In [ ]:
CONSTRAINED_INIT_MODES = ('crystalformer_oracle', 'random_template')
CONSTRAINED_FULL_STEPS = int(ALGO20_FULL_STEPS)
CONSTRAINED_PROJECTION_INTERVAL = int(ALGO20_PROJECTION_INTERVAL)
CONSTRAINED_TAU = float(ALGO20_SAMPLER_TAU)


def sample_template_q(case: GraphCase, payload, *, mode: str, seed: int = SAMPLE_SEED, dtype=None, device=None) -> torch.Tensor:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    dtype = case.gt_frac.dtype if dtype is None else dtype
    device = case.gt_frac.device if device is None else device
    total_dof = int(chart.total_dof)
    if total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'crystalformer_oracle', 'oracle_structure', 'chart_q_ref', 'oracle'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'random_template', 'random', 'rand'}:
        set_seed(int(seed) + 70000 * int(case.graph_id))
        return torch.rand((total_dof,), device=device, dtype=dtype)
    raise ValueError(f'Unsupported constrained init mode: {mode!r}')


def build_template_f0_model(case: GraphCase, payload, *, q0: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q0.reshape(-1), 1.0) if q0.numel() else q0.reshape(-1)
    z_payload = chart.expand_q(q_eval, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f0_model = map_payload_reference_chart_to_model_frame(z_payload, payload)
    return f0_model.detach().clone(), z_payload.detach().clone()


def prepare_constrained_initial_state(case: GraphCase, *, init_mode: str, seed: int = SAMPLE_SEED):
    payload = build_oracle_payload(case)
    q0 = sample_template_q(case, payload, mode=init_mode, seed=seed, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    f0_model, z0_payload = build_template_f0_model(case, payload, q0=q0)
    batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
    t_graph = torch.as_tensor([[1.0]], device=f0_model.device, dtype=f0_model.dtype)
    t_nodes = torch.full((int(f0_model.shape[0]),), 1.0, device=f0_model.device, dtype=f0_model.dtype)
    mode_norm = str(init_mode).strip().lower()
    set_seed(int(seed) + 71000 * int(case.graph_id) + (0 if mode_norm in {'crystalformer_oracle', 'oracle_structure'} else 1))
    f_t, v_t, eps_v, eps_r, r_t = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=f0_model, index=batch_view.batch)
    state_dict = {
        'f_t': f_t.detach().clone(),
        'v_t': v_t.detach().clone(),
        'l_t': case.gt_l_feature.detach().clone().reshape(1, -1),
        'a_t': case.atomic_numbers.detach().clone(),
        'node_index': batch_view.batch.detach().clone(),
        'edge_node_index': batch_view.edge_node_index.detach().clone(),
        'batch': SimpleNamespace(num_atoms=batch_view.num_atoms.detach().clone()),
        'sampling_tdm': runner.model.tdm,
        'score_network': runner.model.score_network,
        'restore_training': False,
        'sampling_time_grid': torch.linspace(1.0, 1e-6, int(CONSTRAINED_FULL_STEPS) + 1, device=f0_model.device, dtype=f0_model.dtype),
    }
    return {
        'payload': payload,
        'q0': q0.detach().clone(),
        'f0_model': f0_model.detach().clone(),
        'z0_payload': z0_payload.detach().clone(),
        'state_dict': state_dict,
        'noise': {'eps_v': eps_v.detach().clone(), 'eps_r': eps_r.detach().clone(), 'r_t': r_t.detach().clone()},
    }


def iter_manual_sampling_times(*, state_dict):
    grid = state_dict['sampling_time_grid']
    for i in range(len(grid) - 1):
        t_now = float(grid[i].item())
        t_next = float(grid[i + 1].item())
        dt = float(t_now - t_next)
        yield i + 1, SimpleNamespace(
            now=SimpleNamespace(
                graph=torch.as_tensor([[t_now]], device=grid.device, dtype=grid.dtype),
                nodes=torch.full((int(state_dict['f_t'].shape[0]),), t_now, device=grid.device, dtype=grid.dtype),
            ),
            next=SimpleNamespace(
                graph=torch.as_tensor([[t_next]], device=grid.device, dtype=grid.dtype),
                nodes=torch.full((int(state_dict['f_t'].shape[0]),), t_next, device=grid.device, dtype=grid.dtype),
                lattice=torch.as_tensor([[t_next]], device=grid.device, dtype=grid.dtype),
            ),
            dt=dt,
            t_next_float=t_next,
        )


def run_constrained_reverse_case(case: GraphCase, *, init_mode: str, use_ppr: bool, seed: int = SAMPLE_SEED):
    prepared = prepare_constrained_initial_state(case, init_mode=init_mode, seed=seed)
    payload = prepared['payload']
    state = prepared['state_dict']
    q_live = prepared['q0'].detach().clone() if prepared['q0'].numel() else prepared['q0']
    projection_trace = []
    started = time.perf_counter()
    for step_idx, times in iter_manual_sampling_times(state_dict=state):
        with torch.no_grad():
            preds_curr = runner.model.score_network(
                t=times.now.graph,
                pos=state['f_t'],
                v=state['v_t'],
                h=state['a_t'],
                l=state['l_t'],
                node_index=state['node_index'],
                edge_node_index=state['edge_node_index'],
            )
            state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_predictor(
                t=times.now.nodes,
                f_t=state['f_t'],
                v_t=state['v_t'],
                pred_v=preds_curr['v'],
                index=state['node_index'],
                dt=times.dt,
            )
            if times.t_next_float >= 1e-3:
                preds_next = runner.model.score_network(
                    t=times.next.graph,
                    pos=state['f_t'],
                    v=state['v_t'],
                    h=state['a_t'],
                    l=state['l_t'],
                    node_index=state['node_index'],
                    edge_node_index=state['edge_node_index'],
                )
                state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_corrector(
                    t=times.next.nodes,
                    f_t=state['f_t'],
                    v_t=state['v_t'],
                    pred_v=preds_next['v'],
                    dt=times.dt,
                    index=state['node_index'],
                    tau=float(CONSTRAINED_TAU),
                )
                state['l_t'] = runner.model._reverse_lattice_sampling_step(
                    t=times.next.lattice,
                    x_t=state['l_t'],
                    pred=preds_next['l'],
                    dt=times.dt,
                    num_atoms=state['batch'].num_atoms,
                )

        if (not use_ppr) or (step_idx % int(CONSTRAINED_PROJECTION_INTERVAL) != 0):
            continue

        scheduled = phase4_schedule(t_value=float(times.t_next_float), base_projection_interval=int(CONSTRAINED_PROJECTION_INTERVAL))
        if not bool(scheduled['use_ppr']) or (step_idx % int(scheduled['projection_interval']) != 0):
            continue
        algo_state = make_algo_state_from_raw(
            f=state['f_t'],
            v=state['v_t'],
            l=state['l_t'],
            atom_types=state['a_t'],
            node_index=state['node_index'],
            edge_node_index=state['edge_node_index'],
            t_graph=times.next.graph,
            t_nodes=times.next.nodes,
        )
        clean_before = clean_prediction(algo_state)
        before_fit = q_only_fit_from_clean(clean_before, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed, q_init=q_live)
        kernel = ppr_kernel_q_witness(
            state=algo_state,
            payload=payload,
            model=runner.model,
            config=replace(config20(M=int(scheduled['M']), lambda0=float(scheduled['lambda0']), q_init_mode=ALGO20_Q_INIT_MODE), anchor_mode=str(scheduled['anchor_mode'])),
            q_init=q_live,
        )
        kernel_tail = kernel.logs[-1] if kernel.logs else {}
        accepted = _accept_phase4_projection(
            before_witness_sin=float(before_fit['fit']['witness_sin']),
            after_witness_sin=float(kernel_tail.get('c_after_witness_sin', float('inf'))),
            soft_anchor_feasible=bool(kernel_tail.get('soft_anchor_feasible', False)),
            rollback=bool(ALGO20_ROLLBACK),
            accept_mode=str(scheduled['accept_mode']),
            min_improvement_ratio=float(scheduled['min_improvement_ratio']),
        )
        if accepted:
            state['f_t'] = kernel.state.f.detach().clone()
            state['v_t'] = kernel.state.v.detach().clone()
            q_live = None if kernel.q_star is None else kernel.q_star.detach().clone()
        projection_trace.append({
            'test': 'algo20_constrained_init_trace',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'init_mode': init_mode,
            'method': 'ppr' if use_ppr else 'baseline',
            'step': int(step_idx),
            't_next': float(times.t_next_float),
            'M': int(scheduled['M']),
            'lambda0': float(scheduled['lambda0']),
            'schedule_label': str(scheduled['schedule_label']),
            'accept_mode': str(scheduled['accept_mode']),
            'soft_anchor_feasible': bool(kernel_tail.get('soft_anchor_feasible', False)),
            'accepted': bool(accepted),
            'c_before_witness': float(before_fit['fit']['witness_sin']),
            'c_after_project_anchor': float(kernel_tail.get('c_after_witness_sin', float('nan'))),
            'witness_rmse_payload': float(kernel_tail.get('witness_rmse_payload', float('nan'))),
            'status': 'INFO',
            'PASS': True,
        })

    elapsed_s = float(time.perf_counter() - started)
    final_state = make_algo_state_from_raw(
        f=state['f_t'],
        v=state['v_t'],
        l=state['l_t'],
        atom_types=state['a_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
        t_graph=torch.as_tensor([[1e-6]], device=state['f_t'].device, dtype=state['f_t'].dtype),
        t_nodes=torch.full((int(state['f_t'].shape[0]),), 1e-6, device=state['f_t'].device, dtype=state['f_t'].dtype),
    )
    final_clean = clean_prediction(final_state)
    final_fit = q_only_fit_from_clean(final_clean, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed, q_init=q_live)
    endpoint = evaluate_arrays(case, state['f_t'], state['l_t'].reshape(-1), state['a_t'])
    feasible_count = sum(bool(row['soft_anchor_feasible']) for row in projection_trace)
    accepted_count = sum(bool(row['accepted']) for row in projection_trace)
    return {
        'test': 'algo20_constrained_init_compare',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'init_mode': init_mode,
        'prior_source': 'crystalformer_oracle' if str(init_mode).strip().lower() in {'crystalformer_oracle', 'oracle_structure'} else 'random_template',
        'method': 'ppr_reverse_from_template_noise' if use_ppr else 'baseline_reverse_from_template_noise',
        'n_steps': int(CONSTRAINED_FULL_STEPS),
        'projection_interval': int(CONSTRAINED_PROJECTION_INTERVAL) if use_ppr else np.nan,
        'runtime_s': elapsed_s,
        'projection_count': len(projection_trace) if use_ppr else np.nan,
        'accepted_fraction': float(accepted_count / max(len(projection_trace), 1)) if use_ppr else np.nan,
        'soft_anchor_feasible_fraction': float(feasible_count / max(len(projection_trace), 1)) if use_ppr else np.nan,
        'final_c_model': float(c_w_ops(state['f_t'], payload).detach().item()),
        'final_witness_sin': float(final_fit['fit']['witness_sin']),
        'frac_rmse': float(endpoint['frac_rmse']),
        'rmse': float(endpoint['rmse']) if endpoint['rmse'] is not None else float('nan'),
        'match': bool(endpoint['match']),
        'valid': bool(endpoint['valid']),
        'PASS': True,
        'status': 'INFO',
    }, projection_trace


constrained_rows = []
constrained_trace_rows = []
for case in GRAPH_CASES:
    for init_mode in CONSTRAINED_INIT_MODES:
        for use_ppr in (False, True):
            try:
                row, trace = run_constrained_reverse_case(case, init_mode=init_mode, use_ppr=use_ppr, seed=int(SAMPLE_SEED))
                constrained_rows.append(row)
                constrained_trace_rows.extend(trace)
            except Exception as exc:
                constrained_rows.append(error_row(
                    'algo20_constrained_init_compare',
                    case.graph_id,
                    exc,
                    'constrained_init_reverse',
                    space_group=case.requested_sg,
                    init_mode=init_mode,
                    method='ppr' if use_ppr else 'baseline',
                ))

constrained_compare_df = dataframe_result('algo20_constrained_init_compare', constrained_rows)
constrained_compare_df = constrained_compare_df.sort_values(['graph', 'init_mode', 'method']).reset_index(drop=True)
display(constrained_compare_df)

constrained_trace_df = dataframe_result('algo20_constrained_init_trace', constrained_trace_rows)
constrained_trace_df = constrained_trace_df.sort_values(['graph', 'init_mode', 'step']).reset_index(drop=True) if not constrained_trace_df.empty else constrained_trace_df
display(constrained_trace_df)


# Debug

Compare matched-time noised-GT states against free PC sampler states.

For saved PC trajectory states, we log:
- q-only witness fit
- joint project-only witness fit with `lambda0 in {0.0, 0.01, 0.1}`
- improvement factor from q-only to joint fit

This is the key diagnostic for separating:
- template/frame incompatibility on free sampler states
- optimizer/schedule weakness
- sampler wiring issues


In [38]:
DEBUG_TRAJECTORY_STEPS = tuple(range(50, int(ALGO20_FULL_STEPS) + 1, 50))
DEBUG_LAMBDA0_VALUES = (0.0, 0.01, 0.1)
DEBUG_GRAPH_IDS = tuple(case.graph_id for case in GRAPH_CASES if int(case.graph_id) in {2, 3, 5}) or tuple(case.graph_id for case in GRAPH_CASES)


def capture_pc_trajectory(case: GraphCase, *, n_steps: int = ALGO20_FULL_STEPS, tau: float = ALGO20_SAMPLER_TAU, seed: int = SAMPLE_SEED):
    set_seed(int(seed) + 50000 * int(case.graph_id) + 17)
    graph_batch = single_graph_batch(case)
    state = runner.model._prepare_csp_sampling(
        batch=graph_batch,
        n_steps=int(n_steps),
        t_start=1.0,
        t_final=1e-6,
    )
    snapshots = []
    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        with torch.no_grad():
            preds_curr = runner.model.score_network(
                t=times.now.graph,
                pos=state['f_t'],
                v=state['v_t'],
                h=state['a_t'],
                l=state['l_t'],
                node_index=state['node_index'],
                edge_node_index=state['edge_node_index'],
            )
            state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_predictor(
                t=times.now.nodes,
                f_t=state['f_t'],
                v_t=state['v_t'],
                pred_v=preds_curr['v'],
                index=state['node_index'],
                dt=times.dt,
            )
            if times.t_next_float >= 1e-3:
                preds_next = runner.model.score_network(
                    t=times.next.graph,
                    pos=state['f_t'],
                    v=state['v_t'],
                    h=state['a_t'],
                    l=state['l_t'],
                    node_index=state['node_index'],
                    edge_node_index=state['edge_node_index'],
                )
                state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_corrector(
                    t=times.next.nodes,
                    f_t=state['f_t'],
                    v_t=state['v_t'],
                    pred_v=preds_next['v'],
                    dt=times.dt,
                    index=state['node_index'],
                    tau=float(tau),
                )
                state['l_t'] = runner.model._reverse_lattice_sampling_step(
                    t=times.next.lattice,
                    x_t=state['l_t'],
                    pred=preds_next['l'],
                    dt=times.dt,
                    num_atoms=state['batch'].num_atoms,
                )
        if step_idx in DEBUG_TRAJECTORY_STEPS:
            snapshots.append({
                'step': int(step_idx),
                't_next': float(times.t_next_float),
                'state': make_algo_state_from_raw(
                    f=state['f_t'],
                    v=state['v_t'],
                    l=state['l_t'],
                    atom_types=state['a_t'],
                    node_index=state['node_index'],
                    edge_node_index=state['edge_node_index'],
                    t_graph=times.next.graph,
                    t_nodes=times.next.nodes,
                ),
            })
    if state['restore_training']:
        state['score_network'].train()
    return snapshots


def debug_state_fit_rows(case: GraphCase, *, state: Algorithm19State, t_value: float, source_kind: str, source_step: int | None = None, seed: int = SAMPLE_SEED) -> list[dict[str, Any]]:
    payload = build_oracle_payload(case)
    clean_f = clean_prediction(state)
    q_only = q_only_fit_from_clean(clean_f, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed)
    rows = []
    for lambda0 in DEBUG_LAMBDA0_VALUES:
        project = ppr_project_step_q_witness(
            state=state,
            payload=payload,
            model=runner.model,
            config=config20(M=1, lambda0=float(lambda0), q_init_mode=ALGO20_Q_INIT_MODE),
            q_init=q_only['fit']['q_star'],
        )
        tail = project.logs[-1] if project.logs else {}
        denom = max(float(project.witness_sin), 1e-12)
        improvement = float(q_only['fit']['witness_sin']) / denom
        rows.append({
            'test': 'algo20_debug_state_witness_fit',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'source_kind': source_kind,
            'source_step': float(source_step) if source_step is not None else np.nan,
            't_value': float(t_value),
            'lambda0': float(lambda0),
            'q_only_witness_sin': float(q_only['fit']['witness_sin']),
            'q_only_witness_rmse': float(q_only['fit']['witness_rmse_payload']),
            'joint_witness_sin': float(project.witness_sin),
            'joint_witness_rmse': float(project.witness_rmse_payload),
            'xi_r_norm': float(tail.get('xi_r_norm', float('nan'))),
            'xi_v_norm': float(tail.get('xi_v_norm', float('nan'))),
            'q_norm': float(tail.get('q_norm', float('nan'))),
            'q_grad_norm': float(tail.get('q_grad_norm', float('nan'))),
            'lambda_eff': float(project.lambda_eff),
            'improvement_factor': float(improvement),
            'PASS': bool(project.witness_sin <= q_only['fit']['witness_sin'] + 1e-12),
            'status': 'PASS' if bool(project.witness_sin <= q_only['fit']['witness_sin'] + 1e-12) else 'WARN',
        })
    return rows


debug_rows = []
for case in GRAPH_CASES:
    if int(case.graph_id) not in set(DEBUG_GRAPH_IDS):
        continue
    try:
        snapshots = capture_pc_trajectory(case, n_steps=int(ALGO20_FULL_STEPS), tau=float(ALGO20_SAMPLER_TAU), seed=int(SAMPLE_SEED))
    except Exception as exc:
        debug_rows.append(error_row(
            'algo20_debug_state_witness_fit',
            case.graph_id,
            exc,
            'capture_pc_trajectory',
            space_group=case.requested_sg,
        ))
        continue

    for snap in snapshots:
        try:
            debug_rows.extend(debug_state_fit_rows(
                case,
                state=snap['state'],
                t_value=float(snap['t_next']),
                source_kind='pc_state',
                source_step=int(snap['step']),
                seed=int(SAMPLE_SEED),
            ))
        except Exception as exc:
            debug_rows.append(error_row(
                'algo20_debug_state_witness_fit',
                case.graph_id,
                exc,
                'pc_state_debug_fit',
                space_group=case.requested_sg,
                t_value=float(snap['t_next']),
                source_kind='pc_state',
                source_step=int(snap['step']),
            ))

        try:
            gt_state, _ = make_noisy_state(case, t_value=float(snap['t_next']), seed=int(SAMPLE_SEED))
            debug_rows.extend(debug_state_fit_rows(
                case,
                state=gt_state,
                t_value=float(snap['t_next']),
                source_kind='gt_noised',
                source_step=int(snap['step']),
                seed=int(SAMPLE_SEED),
            ))
        except Exception as exc:
            debug_rows.append(error_row(
                'algo20_debug_state_witness_fit',
                case.graph_id,
                exc,
                'gt_state_debug_fit',
                space_group=case.requested_sg,
                t_value=float(snap['t_next']),
                source_kind='gt_noised',
                source_step=int(snap['step']),
            ))

debug_state_df = dataframe_result('algo20_debug_state_witness_fit', debug_rows)
debug_state_df = debug_state_df.sort_values(['graph', 'source_step', 'source_kind', 'lambda0']).reset_index(drop=True)
display(debug_state_df)


,test,graph,space_group,source_kind,source_step,t_value,lambda0,q_only_witness_sin,q_only_witness_rmse,joint_witness_sin,joint_witness_rmse,xi_r_norm,xi_v_norm,q_norm,q_grad_norm,lambda_eff,improvement_factor,PASS,status
0,algo20_debug_state_witness_fit,2,4,gt_noised,50.0,9.166667e-01,0.00,0.025667,0.051594,8.223597e-04,0.009131,0.041320,0.034711,2.807270,0.001001,1.000000e-06,31.211773,True,PASS
1,algo20_debug_state_witness_fit,2,4,gt_noised,50.0,9.166667e-01,0.01,0.025667,0.051594,8.057095e-04,0.009038,0.040849,0.034557,2.808363,0.001057,2.055302e-03,31.856774,True,PASS
2,algo20_debug_state_witness_fit,2,4,gt_noised,50.0,9.166667e-01,0.10,0.025667,0.051594,6.931740e-04,0.008383,0.042621,0.035185,2.808335,0.000841,2.055302e-02,37.028659,True,PASS
3,algo20_debug_state_witness_fit,2,4,pc_state,50.0,9.166667e-01,0.00,0.162335,0.135901,1.081527e-01,0.109495,0.078298,0.062201,3.003356,0.004520,1.000000e-06,1.500979,True,PASS
4,algo20_debug_state_witness_fit,2,4,pc_state,50.0,9.166667e-01,0.01,0.162335,0.135901,1.080243e-01,0.109339,0.076831,0.065274,3.001653,0.003571,2.055302e-03,1.502764,True,PASS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,algo20_debug_state_witness_fit,2,4,gt_noised,600.0,1.000000e-06,0.01,0.000035,0.001889,1.154607e-07,0.000108,0.036239,0.000003,2.969452,0.000150,1.000000e-06,305.151646,True,PASS
68,algo20_debug_state_witness_fit,2,4,gt_noised,600.0,1.000000e-06,0.10,0.000035,0.001889,1.154607e-07,0.000108,0.036239,0.000003,2.969452,0.000150,1.000000e-06,305.151646,True,PASS
69,algo20_debug_state_witness_fit,2,4,pc_state,600.0,1.000000e-06,0.00,0.118008,0.113744,1.177209e-01,0.113596,0.952859,0.006093,3.062725,0.000122,1.000000e-06,1.002436,True,PASS
70,algo20_debug_state_witness_fit,2,4,pc_state,600.0,1.000000e-06,0.01,0.118008,0.113744,1.177209e-01,0.113596,0.952859,0.006093,3.062725,0.000122,1.000000e-06,1.002436,True,PASS


## Lightweight mismatch debugging

These tests are meant to answer a narrower question than full PPR sampling:

1. Does species-aware assignment alone explain most of the witness error?
2. Are free PC clean states actually far from the GT basin in model frame?
3. Does a nearby alternative template explain the PC state better than the oracle template?
4. If we start reverse dynamics from a GT-noised state instead of the prior, does the witness stay in a much better basin?


In [ ]:
try:
    from scipy.optimize import linear_sum_assignment
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
    linear_sum_assignment = None

LIGHT_DEBUG_STEPS = DEBUG_TRAJECTORY_STEPS
LIGHT_DEBUG_GRAPH_IDS = DEBUG_GRAPH_IDS


def torus_pairwise_cost(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    diff = wrapdiff(a[:, None, :], b[None, :, :])
    return diff.square().mean(dim=-1)


def species_preserving_assignment(
    source_frac: torch.Tensor,
    target_frac: torch.Tensor,
    source_species: torch.Tensor,
    target_species: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    if source_frac.shape != target_frac.shape:
        raise ValueError(f'shape mismatch: {tuple(source_frac.shape)} vs {tuple(target_frac.shape)}')
    source_species = torch.as_tensor(source_species, device=source_frac.device, dtype=torch.long).reshape(-1)
    target_species = torch.as_tensor(target_species, device=target_frac.device, dtype=torch.long).reshape(-1)
    if source_species.shape != target_species.shape:
        raise ValueError('species shape mismatch')
    assigned_source = torch.empty_like(source_frac)
    permutation = torch.empty((target_frac.shape[0],), device=source_frac.device, dtype=torch.long)
    for species in sorted(set(int(v) for v in source_species.tolist())):
        src_idx = torch.nonzero(source_species == int(species), as_tuple=False).reshape(-1)
        tgt_idx = torch.nonzero(target_species == int(species), as_tuple=False).reshape(-1)
        if src_idx.numel() != tgt_idx.numel():
            raise ValueError(f'species multiplicity mismatch for Z={species}: {int(src_idx.numel())} vs {int(tgt_idx.numel())}')
        if src_idx.numel() == 0:
            continue
        cost = torus_pairwise_cost(source_frac[src_idx], target_frac[tgt_idx]).detach().cpu().numpy()
        if HAVE_SCIPY:
            row_ind, col_ind = linear_sum_assignment(cost)
        else:
            remaining = list(range(cost.shape[1]))
            row_ind = []
            col_ind = []
            for r in range(cost.shape[0]):
                c = min(remaining, key=lambda cc: float(cost[r, cc]))
                row_ind.append(r)
                col_ind.append(c)
                remaining.remove(c)
        for r, c in zip(row_ind, col_ind):
            src_global = src_idx[int(r)]
            tgt_global = tgt_idx[int(c)]
            assigned_source[tgt_global] = source_frac[src_global]
            permutation[tgt_global] = src_global
    return assigned_source, permutation


def build_wrong_spacegroup_payload(case: GraphCase, payload) -> Any | None:
    letters = list(payload.wyckoff_letters)
    atom_types = np.asarray(payload.anchor_atomic_numbers, dtype=int).tolist()
    for delta in range(1, 33):
        for sign in (-1, 1):
            candidate_sg = int(payload.spacegroup) + sign * delta
            if candidate_sg < 1 or candidate_sg > 230 or candidate_sg == int(payload.spacegroup):
                continue
            try:
                wrong = build_diffcsppp_payload_from_syminfo(
                    spacegroup_number=candidate_sg,
                    wyckoff_letters=letters,
                    atom_types=atom_types,
                )
            except Exception:
                continue
            if int(wrong.num_atoms) != int(payload.num_atoms):
                continue
            return replace(
                wrong,
                anchor_frac_coords=np.asarray(payload.anchor_frac_coords, dtype=float).copy(),
                debug_info={**(payload.debug_info or {}), **(wrong.debug_info or {}), 'source': 'wrong_spacegroup_same_letters'},
            )
    return None


def best_template_fit(clean_f: torch.Tensor, case: GraphCase, oracle_payload, *, seed: int) -> dict[str, Any]:
    candidates = [('oracle', oracle_payload)]
    wrong_payload = build_wrong_spacegroup_payload(case, oracle_payload)
    if wrong_payload is not None:
        candidates.append(('nearby_wrong_sg', wrong_payload))
    best_name = None
    best_fit = None
    best_payload = None
    for name, payload in candidates:
        fit = q_only_fit_from_clean(clean_f, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed)
        if best_fit is None or float(fit['fit']['witness_sin']) < float(best_fit['fit']['witness_sin']):
            best_name = name
            best_fit = fit
            best_payload = payload
    return {
        'best_template_name': best_name,
        'best_fit': best_fit,
        'best_payload': best_payload,
        'num_templates': len(candidates),
    }


def short_reverse_from_gt_noised(case: GraphCase, *, t_start: float, num_steps: int = 50, tau: float = ALGO20_SAMPLER_TAU, seed: int = SAMPLE_SEED):
    state, _ = make_noisy_state(case, t_value=float(t_start), seed=int(seed))
    t_grid = torch.linspace(float(t_start), 1e-6, int(num_steps) + 1, device=state.f.device, dtype=state.f.dtype)
    current_f = state.f.detach().clone()
    current_v = state.v.detach().clone()
    current_l = case.gt_l_feature.detach().clone().reshape(1, -1).to(device=state.f.device, dtype=state.f.dtype)
    node_index = state.node_index.detach().clone()
    edge_node_index = state.edge_node_index.detach().clone()
    atom_types = state.atom_types.detach().clone()
    for idx in range(int(num_steps)):
        t_now = float(t_grid[idx].item())
        t_next = float(t_grid[idx + 1].item())
        dt = float(t_now - t_next)
        t_graph_now = torch.as_tensor([[t_now]], device=current_f.device, dtype=current_f.dtype)
        t_nodes_now = torch.full((current_f.shape[0],), t_now, device=current_f.device, dtype=current_f.dtype)
        with torch.no_grad():
            preds_now = runner.model.score_network(
                t=t_graph_now,
                pos=current_f,
                v=current_v,
                h=atom_types,
                l=current_l,
                node_index=node_index,
                edge_node_index=edge_node_index,
            )
            current_f, current_v = runner.model.tdm.reverse_step_predictor(
                t=t_nodes_now,
                f_t=current_f,
                v_t=current_v,
                pred_v=preds_now['v'],
                index=node_index,
                dt=dt,
            )
            if t_next >= 1e-3:
                t_graph_next = torch.as_tensor([[t_next]], device=current_f.device, dtype=current_f.dtype)
                t_nodes_next = torch.full((current_f.shape[0],), t_next, device=current_f.device, dtype=current_f.dtype)
                preds_next = runner.model.score_network(
                    t=t_graph_next,
                    pos=current_f,
                    v=current_v,
                    h=atom_types,
                    l=current_l,
                    node_index=node_index,
                    edge_node_index=edge_node_index,
                )
                current_f, current_v = runner.model.tdm.reverse_step_corrector(
                    t=t_nodes_next,
                    f_t=current_f,
                    v_t=current_v,
                    pred_v=preds_next['v'],
                    dt=dt,
                    index=node_index,
                    tau=float(tau),
                )
    return make_algo_state_from_raw(
        f=current_f,
        v=current_v,
        l=current_l,
        atom_types=atom_types,
        node_index=node_index,
        edge_node_index=edge_node_index,
        t_graph=torch.as_tensor([[float(t_grid[-1].item())]], device=current_f.device, dtype=current_f.dtype),
        t_nodes=torch.full((current_f.shape[0],), float(t_grid[-1].item()), device=current_f.device, dtype=current_f.dtype),
    )


light_rows = []
for case in GRAPH_CASES:
    if int(case.graph_id) not in set(LIGHT_DEBUG_GRAPH_IDS):
        continue
    oracle_payload = build_oracle_payload(case)
    try:
        pc_snapshots = capture_pc_trajectory(case, n_steps=int(ALGO20_FULL_STEPS), tau=float(ALGO20_SAMPLER_TAU), seed=int(SAMPLE_SEED))
    except Exception as exc:
        light_rows.append(error_row(
            'algo20_lightweight_debug',
            case.graph_id,
            exc,
            'capture_pc_trajectory_light',
            space_group=case.requested_sg,
        ))
        continue

    for snap in pc_snapshots:
        try:
            pc_clean = clean_prediction(snap['state'])
            pc_q_oracle = q_only_fit_from_clean(pc_clean, oracle_payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=SAMPLE_SEED)
            oracle_chart = algo19_backend._get_wyckoff_dof_chart(oracle_payload)
            pc_z_payload = map_model_to_payload_reference_chart(pc_clean, oracle_payload)
            pc_z_sym = oracle_chart.expand_q(pc_q_oracle['fit']['q_star'], device=pc_clean.device, dtype=pc_clean.dtype)
            pc_z_aligned, _ = species_preserving_assignment(
                pc_z_payload,
                pc_z_sym,
                case.atomic_numbers,
                torch.as_tensor(oracle_payload.expanded_atomic_numbers, device=pc_clean.device, dtype=torch.long),
            )
            assigned_witness_sin = float(witness_torus_sin_loss(pc_z_aligned, pc_z_sym).detach().item())
            assigned_witness_rmse = float(torus_rmse(pc_z_aligned, pc_z_sym).detach().item())

            gt_aligned, _ = species_preserving_assignment(
                pc_clean,
                case.gt_frac,
                case.atomic_numbers,
                case.atomic_numbers,
            )
            gt_aligned_rmse = float(torus_rmse(pc_clean, gt_aligned).detach().item())

            template_fit = best_template_fit(pc_clean, case, oracle_payload, seed=SAMPLE_SEED)
            short_state = short_reverse_from_gt_noised(case, t_start=float(snap['t_next']), num_steps=min(50, max(10, int(snap['step']))), tau=float(ALGO20_SAMPLER_TAU), seed=int(SAMPLE_SEED))
            short_clean = clean_prediction(short_state)
            short_fit = q_only_fit_from_clean(short_clean, oracle_payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=SAMPLE_SEED)

            light_rows.append({
                'test': 'algo20_lightweight_debug',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                'step': int(snap['step']),
                't_value': float(snap['t_next']),
                'q_only_witness_sin': float(pc_q_oracle['fit']['witness_sin']),
                'q_only_witness_rmse': float(pc_q_oracle['fit']['witness_rmse_payload']),
                'assigned_witness_sin': float(assigned_witness_sin),
                'assigned_witness_rmse': float(assigned_witness_rmse),
                'assignment_improvement_factor': float(pc_q_oracle['fit']['witness_sin'] / max(assigned_witness_sin, 1e-12)),
                'gt_aligned_rmse': float(gt_aligned_rmse),
                'best_template_name': str(template_fit['best_template_name']),
                'best_template_witness_sin': float(template_fit['best_fit']['fit']['witness_sin']),
                'num_templates_tested': int(template_fit['num_templates']),
                'short_gt_reverse_witness_sin': float(short_fit['fit']['witness_sin']),
                'short_gt_reverse_witness_rmse': float(short_fit['fit']['witness_rmse_payload']),
                'PASS': bool(True),
                'status': 'INFO',
            })
        except Exception as exc:
            light_rows.append(error_row(
                'algo20_lightweight_debug',
                case.graph_id,
                exc,
                'lightweight_debug_eval',
                space_group=case.requested_sg,
                step=int(snap['step']),
                t_value=float(snap['t_next']),
            ))

light_debug_df = dataframe_result('algo20_lightweight_debug', light_rows)
light_debug_df = light_debug_df.sort_values(['graph', 'step']).reset_index(drop=True)
display(light_debug_df)


### Assignment sanity audit

This cell checks whether the species-aware assignment diagnostic is genuinely aligning
PC payload rows to the oracle witness structure, or accidentally comparing a tensor to itself.


In [ ]:

try:
    species_preserving_assignment
except NameError:
    def _audit_torus_pairwise_cost(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        diff = wrapdiff(a[:, None, :], b[None, :, :])
        return diff.square().mean(dim=-1)

    def species_preserving_assignment(
        source_frac: torch.Tensor,
        target_frac: torch.Tensor,
        source_species: torch.Tensor,
        target_species: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if source_frac.shape != target_frac.shape:
            raise ValueError(f'shape mismatch: {tuple(source_frac.shape)} vs {tuple(target_frac.shape)}')
        source_species = torch.as_tensor(source_species, device=source_frac.device, dtype=torch.long).reshape(-1)
        target_species = torch.as_tensor(target_species, device=target_frac.device, dtype=torch.long).reshape(-1)
        if source_species.shape != target_species.shape:
            raise ValueError('species shape mismatch')
        assigned_source = torch.empty_like(source_frac)
        permutation = torch.empty((target_frac.shape[0],), device=source_frac.device, dtype=torch.long)
        have_scipy = 'linear_sum_assignment' in globals() and globals().get('linear_sum_assignment') is not None
        for species in sorted(set(int(v) for v in source_species.tolist())):
            src_idx = torch.nonzero(source_species == int(species), as_tuple=False).reshape(-1)
            tgt_idx = torch.nonzero(target_species == int(species), as_tuple=False).reshape(-1)
            if src_idx.numel() != tgt_idx.numel():
                raise ValueError(f'species multiplicity mismatch for Z={species}: {int(src_idx.numel())} vs {int(tgt_idx.numel())}')
            if src_idx.numel() == 0:
                continue
            cost = _audit_torus_pairwise_cost(source_frac[src_idx], target_frac[tgt_idx]).detach().cpu().numpy()
            if have_scipy:
                row_ind, col_ind = linear_sum_assignment(cost)
            else:
                remaining = list(range(cost.shape[1]))
                row_ind = []
                col_ind = []
                for r in range(cost.shape[0]):
                    c = min(remaining, key=lambda cc: float(cost[r, cc]))
                    row_ind.append(r)
                    col_ind.append(c)
                    remaining.remove(c)
            for r, c in zip(row_ind, col_ind):
                src_global = src_idx[int(r)]
                tgt_global = tgt_idx[int(c)]
                assigned_source[tgt_global] = source_frac[src_global]
                permutation[tgt_global] = src_global
        return assigned_source, permutation

ASSIGNMENT_AUDIT_GRAPH_IDS = tuple(globals().get('LIGHT_DEBUG_GRAPH_IDS', globals().get('DEBUG_GRAPH_IDS', tuple(case.graph_id for case in GRAPH_CASES))))
ASSIGNMENT_AUDIT_GRAPH = next((case.graph_id for case in GRAPH_CASES if int(case.graph_id) in set(ASSIGNMENT_AUDIT_GRAPH_IDS)), GRAPH_CASES[0].graph_id)
ASSIGNMENT_AUDIT_STEPS = tuple(globals().get('DEBUG_TRAJECTORY_STEPS', tuple(range(50, int(ALGO20_FULL_STEPS) + 1, 50))))
ASSIGNMENT_AUDIT_STEP = ASSIGNMENT_AUDIT_STEPS[0]


def assignment_audit_row(case: GraphCase, *, step_target: int, seed: int = SAMPLE_SEED) -> dict[str, Any]:
    payload = build_oracle_payload(case)
    snapshots = capture_pc_trajectory(case, n_steps=int(ALGO20_FULL_STEPS), tau=float(ALGO20_SAMPLER_TAU), seed=int(seed))
    snap = next(item for item in snapshots if int(item['step']) == int(step_target))
    clean_f = clean_prediction(snap['state'])
    q_fit = q_only_fit_from_clean(clean_f, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    z_proj = chart.expand_q(q_fit['fit']['q_star'], device=clean_f.device, dtype=clean_f.dtype)
    assigned_target, permutation = species_preserving_assignment(
        z_payload,
        z_proj,
        case.atomic_numbers,
        torch.as_tensor(payload.expanded_atomic_numbers, device=clean_f.device, dtype=torch.long),
    )
    unique_perm = bool(len(set(int(v) for v in permutation.detach().cpu().tolist())) == int(permutation.numel()))
    species_src = torch.as_tensor(case.atomic_numbers, device=clean_f.device, dtype=torch.long).reshape(-1)
    species_tgt = torch.as_tensor(payload.expanded_atomic_numbers, device=clean_f.device, dtype=torch.long).reshape(-1)
    assigned_src_species = species_src.detach().cpu()[permutation.detach().cpu()].tolist()
    target_species = species_tgt.detach().cpu().tolist()
    raw_rmse = float(torus_rmse(z_payload, z_proj).detach().item())
    assigned_rmse = float(torus_rmse(assigned_target, z_proj).detach().item())
    self_rmse_source = float(torus_rmse(assigned_target, assigned_target).detach().item())
    self_rmse_target = float(torus_rmse(z_proj, z_proj).detach().item())
    delta = wrapdiff(assigned_target, z_proj)
    return {
        'test': 'algo20_assignment_audit',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'step': int(step_target),
        't_value': float(snap['t_next']),
        'z_payload_shape': tuple(z_payload.shape),
        'z_proj_shape': tuple(z_proj.shape),
        'assignment': permutation.detach().cpu().tolist(),
        'unique_assignment': unique_perm,
        'assigned_source_species': assigned_src_species,
        'target_species': target_species,
        'raw_rmse': raw_rmse,
        'assigned_rmse': assigned_rmse,
        'self_rmse_source': self_rmse_source,
        'self_rmse_target': self_rmse_target,
        'delta_abs_max': float(delta.abs().max().detach().item()),
        'delta_sq_mean': float(delta.square().mean().detach().item()),
        'q_only_witness_sin': float(q_fit['fit']['witness_sin']),
        'assigned_witness_sin': float(witness_torus_sin_loss(assigned_target, z_proj).detach().item()),
        'PASS': True,
        'status': 'INFO',
    }


audit_rows = []
for case in GRAPH_CASES:
    if int(case.graph_id) != int(ASSIGNMENT_AUDIT_GRAPH):
        continue
    try:
        audit_rows.append(assignment_audit_row(case, step_target=int(ASSIGNMENT_AUDIT_STEP), seed=int(SAMPLE_SEED)))
    except Exception as exc:
        audit_rows.append(error_row(
            'algo20_assignment_audit',
            case.graph_id,
            exc,
            'assignment_audit',
            space_group=case.requested_sg,
            step=int(ASSIGNMENT_AUDIT_STEP),
        ))

assignment_audit_df = dataframe_result('algo20_assignment_audit', audit_rows)
display(assignment_audit_df)


### Multi-template scan

For saved free-PC states, scan a broader set of nearby compatible templates and rank them by
q-only witness fit. This checks whether the sampler is drifting into a different template basin,
or whether nothing nearby explains the PC state well.


In [ ]:
TEMPLATE_SCAN_GRAPH_IDS = ASSIGNMENT_AUDIT_GRAPH_IDS if 'ASSIGNMENT_AUDIT_GRAPH_IDS' in globals() else DEBUG_GRAPH_IDS
TEMPLATE_SCAN_STEPS = tuple(globals().get('DEBUG_TRAJECTORY_STEPS', tuple(range(50, int(ALGO20_FULL_STEPS) + 1, 50))))
TEMPLATE_SCAN_MAX_CANDIDATES = 24
TEMPLATE_SCAN_TOP_K = 5


def build_nearby_template_candidates(case: GraphCase, oracle_payload, *, max_candidates: int = TEMPLATE_SCAN_MAX_CANDIDATES):
    letters = list(oracle_payload.wyckoff_letters)
    atom_types = np.asarray(oracle_payload.anchor_atomic_numbers, dtype=int).tolist()
    candidates = [('oracle', oracle_payload)]
    seen = {(int(oracle_payload.spacegroup), tuple(letters), tuple(atom_types), 'oracle')}

    # Same letters/species, nearby space groups first.
    for delta in range(1, 65):
        for sign in (-1, 1):
            candidate_sg = int(oracle_payload.spacegroup) + sign * delta
            if candidate_sg < 1 or candidate_sg > 230:
                continue
            key = (candidate_sg, tuple(letters), tuple(atom_types), 'nearby_sg')
            if key in seen:
                continue
            try:
                payload = build_diffcsppp_payload_from_syminfo(
                    spacegroup_number=candidate_sg,
                    wyckoff_letters=letters,
                    atom_types=atom_types,
                )
            except Exception:
                continue
            if int(payload.num_atoms) != int(oracle_payload.num_atoms):
                continue
            payload = replace(
                payload,
                anchor_frac_coords=np.asarray(oracle_payload.anchor_frac_coords, dtype=float).copy(),
                debug_info={**(oracle_payload.debug_info or {}), **(payload.debug_info or {}), 'source': 'nearby_sg_scan'},
            )
            candidates.append((f'sg_{candidate_sg}', payload))
            seen.add(key)
            if len(candidates) >= int(max_candidates):
                return candidates

    # Species rotations as a secondary cheap probe.
    atom_types_arr = np.asarray(atom_types, dtype=int)
    if len(set(atom_types_arr.tolist())) > 1:
        for shift in range(1, len(atom_types_arr)):
            rotated = np.roll(atom_types_arr, shift)
            key = (int(oracle_payload.spacegroup), tuple(letters), tuple(rotated.tolist()), f'rot_{shift}')
            if key in seen:
                continue
            try:
                payload = build_diffcsppp_payload_from_syminfo(
                    spacegroup_number=int(oracle_payload.spacegroup),
                    wyckoff_letters=letters,
                    atom_types=rotated.tolist(),
                )
            except Exception:
                continue
            if int(payload.num_atoms) != int(oracle_payload.num_atoms):
                continue
            payload = replace(
                payload,
                anchor_frac_coords=np.asarray(oracle_payload.anchor_frac_coords, dtype=float).copy(),
                debug_info={**(oracle_payload.debug_info or {}), **(payload.debug_info or {}), 'source': 'species_rotation_scan'},
            )
            candidates.append((f'species_rot_{shift}', payload))
            seen.add(key)
            if len(candidates) >= int(max_candidates):
                return candidates
    return candidates


def template_scan_for_state(case: GraphCase, *, state: Algorithm19State, step: int, t_value: float, seed: int = SAMPLE_SEED):
    oracle_payload = build_oracle_payload(case)
    clean_f = clean_prediction(state)
    candidates = build_nearby_template_candidates(case, oracle_payload)
    rows = []
    for template_name, payload in candidates:
        try:
            fit = q_only_fit_from_clean(clean_f, payload, case=case, q_init_mode=ALGO20_Q_INIT_MODE, seed=seed)
            rows.append({
                'test': 'algo20_template_scan',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                'step': int(step),
                't_value': float(t_value),
                'template_name': str(template_name),
                'template_space_group': int(payload.spacegroup),
                'num_atoms': int(payload.num_atoms),
                'witness_sin': float(fit['fit']['witness_sin']),
                'witness_rmse': float(fit['fit']['witness_rmse_payload']),
                'PASS': True,
                'status': 'INFO',
            })
        except Exception as exc:
            rows.append(error_row(
                'algo20_template_scan',
                case.graph_id,
                exc,
                'template_scan_fit',
                space_group=case.requested_sg,
                step=int(step),
                t_value=float(t_value),
                template_name=str(template_name),
                template_space_group=int(payload.spacegroup),
            ))
    return rows


template_scan_rows = []
for case in GRAPH_CASES:
    if int(case.graph_id) not in set(TEMPLATE_SCAN_GRAPH_IDS):
        continue
    try:
        snapshots = capture_pc_trajectory(case, n_steps=int(ALGO20_FULL_STEPS), tau=float(ALGO20_SAMPLER_TAU), seed=int(SAMPLE_SEED))
    except Exception as exc:
        template_scan_rows.append(error_row(
            'algo20_template_scan',
            case.graph_id,
            exc,
            'template_scan_capture',
            space_group=case.requested_sg,
        ))
        continue
    for snap in snapshots:
        if int(snap['step']) not in set(TEMPLATE_SCAN_STEPS):
            continue
        template_scan_rows.extend(template_scan_for_state(
            case,
            state=snap['state'],
            step=int(snap['step']),
            t_value=float(snap['t_next']),
            seed=int(SAMPLE_SEED),
        ))

template_scan_df = dataframe_result('algo20_template_scan', template_scan_rows)
template_scan_df = template_scan_df.sort_values(['graph', 'step', 'witness_sin', 'template_name']).reset_index(drop=True)
display(template_scan_df)

template_scan_top_df = template_scan_df[template_scan_df.get('status', pd.Series(dtype=object)).eq('INFO')].copy() if not template_scan_df.empty else template_scan_df.copy()
if not template_scan_top_df.empty:
    template_scan_top_df = (
        template_scan_top_df
        .sort_values(['graph', 'step', 'witness_sin', 'template_name'])
        .groupby(['graph', 'step'], as_index=False, group_keys=False)
        .head(int(TEMPLATE_SCAN_TOP_K))
        .reset_index(drop=True)
    )
display(template_scan_top_df)


## Notes

- `oracle_structure` q-init is an upper-bound ablation. The fair default for CSP-style testing is `random`.
- The fixed-template witness lives in payload/template frame through the current bridge `B`.
- The full sampler uses rollback during debugging: a PPR update is accepted only when the soft witness anchor is feasible.
